# Eburon TTS Training - Dutch Flemish & Tagalog

Train custom TTS models using Unsloth


In [ ]:
# Install Dependencies
!pip install -q unsloth transformers datasets torch soundfile scipy
!pip install -q accelerate peft bitsandbytes

In [ ]:
# Download Dataset
LANGUAGE = "nl"  # Change to "tl" for Tagalog
MAX_HOURS = 5

from datasets import load_dataset
import soundfile as sf

print(f"Downloading {LANGUAGE} dataset...")
ds = load_dataset("common_voice", LANGUAGE, split="train")
print(f"Total samples: {len(ds)}")

samples_needed = min(MAX_HOURS * 1000, len(ds))
train_data = ds.select(range(samples_needed))

audio_data = []
for i, example in enumerate(train_data):
    audio = example["audio"]
    audio_data.append({
        "audio": audio["array"],
        "text": example["sentence"],
        "sample_rate": audio["sampling_rate"]
    })
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1} samples")

print(f"Downloaded {len(audio_data)} audio samples")

In [ ]:
# Train Model
MODEL_SIZE = "0.6B"
EPOCHS = 3

from unsloth import FastLanguageModel
import torch

model_name = f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base"
print(f"Loading model: {model_name}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

print("Model ready! Add training loop here.")

In [ ]:
# Save Model
OUTPUT_NAME = "eburon-tts-dutch"

model.save_pretrained(OUTPUT_NAME)
tokenizer.save_pretrained(OUTPUT_NAME)
print(f"Model saved to: {OUTPUT_NAME}")

import shutil
shutil.make_archive(OUTPUT_NAME, 'zip', OUTPUT_NAME)
print(f"Zipped: {OUTPUT_NAME}.zip")

In [ ]:
# Test Model
test_text = "Hello! This is a test."

from mlx_audio.tts import load
import soundfile as sf

trained_model = load(f"./{OUTPUT_NAME}")
result = list(trained_model.generate(text=test_text))[-1]

output_file = "test_output.wav"
sf.write(output_file, result.audio, result.sample_rate)

print(f"Generated: {output_file}")

from IPython.display import Audio
Audio(output_file)